In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.preprocess import PreprocessConfig, build_manifest, make_resize_cache

cfg = PreprocessConfig(
    raw_root=Path("../data/raw"),
    train_dirname="Train",
    test_dirname="Test",
    processed_dir=Path("../data/processed"),
    manifest_path=Path("../data/processed/manifest.csv"),
    label2id_path=Path("../data/processed/label2id.json"),
    val_size=0.15,
    seed=42,
    save_resized=True,
    resized_dir=Path("../data/processed/resized_224"),
    img_size=(224, 224),
    skip_broken_images=True,
)

manifest = build_manifest(cfg)
manifest["split"].value_counts()

split
train    1903
val       336
test      118
Name: count, dtype: int64

In [3]:
manifest = make_resize_cache(cfg)
manifest.head()

,path,label,split,label_id,path_resized
0,..\data\raw\Train\pigmented benign keratosis\I...,pigmented benign keratosis,train,5,..\data\processed\resized_224\train_pigmented ...
1,..\data\raw\Train\basal cell carcinoma\ISIC_00...,basal cell carcinoma,train,1,..\data\processed\resized_224\train_basal cell...
2,..\data\raw\Train\squamous cell carcinoma\ISIC...,squamous cell carcinoma,train,7,..\data\processed\resized_224\train_squamous c...
3,..\data\raw\Train\nevus\ISIC_0000324.jpg,nevus,train,4,..\data\processed\resized_224\train_nevus_3.jpg
4,..\data\raw\Train\squamous cell carcinoma\ISIC...,squamous cell carcinoma,train,7,..\data\processed\resized_224\train_squamous c...


In [5]:
from src.embeddings import EmbeddingConfig, extract_embeddings

ecfg = EmbeddingConfig(
    manifest_path=Path("../data/processed/manifest.csv"),
    out_dir=Path("../data/processed/features_raw"),
    batch_size=32,
)

extract_embeddings(ecfg)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Administrator/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [6]:
from src.qfeatures import QFeatureConfig, make_quantum_features

qcfg = QFeatureConfig(
    in_dir=Path("../data/processed/features_raw"),
    out_dir=Path("../data/processed/features_q"),
    n_components=8,
    seed=42,
    clip_z=3.0,
)

make_quantum_features(qcfg)

import numpy as np
Xtr = np.load("../data/processed/features_q/X_train.npy")
Xtr.shape, (float(Xtr.min()), float(Xtr.max()))

((1903, 8), (-3.1415927410125732, 3.1415927410125732))